### Google A2A Protocol Demo:

Demo Abstract:


Setup:

pip install uvicorn a2a-sdk python-dotenv

git clone https://github.com/a2aproject/a2a-samples.git -b main --depth 1

python -c "import a2a; print('A2A SDK imported successfully')"

### Agent Skills and Agent Card

Some discretions between https://a2a-protocol.org/latest/tutorials/python/3-agent-skills-and-card/#agent-card and its lib src.

In [ ]:
from a2a.types import AgentSkill, AgentCard, AgentInterface, AgentCapabilities


skill = AgentSkill( # min requires id, name des, tags
    description='just returns hello world',
    id='1',
    name='hello world',
    tags=['hello world'],
)
# skill = AgentSkill(descritpion, examples, id, input_modes, output_modes, security, tags)


card = AgentCard(
    name='hello world agent',
    description='just a hello world agent',

    capabilities= AgentCapabilities(
        streaming=True
    ),
    skills=[skill],

    url='http://localhost:9999/',
    additional_interfaces=[
        AgentInterface(
            url='http://localhost:9999',
            transport='JSONRPC',
        )
    ],
    
    version='1.0.0',

    default_input_modes=['text'],
    default_output_modes=['text'],
)


### Agent Executor


In [ ]:
class HelloWorldAgent:
    """
    """

    async def invoke(self) -> str:
        return "hello world!"
    
  
from a2a.server.agent_execution import RequestContext
from a2a.server.events import EventQueue
from a2a.types import TaskStatusUpdateEvent, TaskStatus, TaskState, TaskArtifactUpdateEvent

from a2a.utils.task import new_task
from a2a.utils.message import new_agent_text_message
from a2a.utils.artifact import new_text_artifact


class HelloWorldAgentExecutor:
    """
    """

    def __init__(self) -> None:
        self.agent = HelloWorldAgent()
    
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task(context.message) # pyright: ignore[reportArgumentType]

        await event_queue.enqueue_event(task)
        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id, # pyright: ignore[reportArgumentType]
                context_id=context.context_id, # pyright: ignore[reportArgumentType]
                status=TaskStatus(
                    state=TaskState.working, # state=TaskState.TASK_STATE_WORKING,
                    message=new_agent_text_message('Processing request...'),
                ),
                final=False
            )
        )
        result = await self.agent.invoke()
        await event_queue.enqueue_event(
            TaskArtifactUpdateEvent(
                task_id=context.task_id, # pyright: ignore[reportArgumentType]
                context_id=context.context_id, # pyright: ignore[reportArgumentType]
                artifact=new_text_artifact(name='result', text=result),
            )
        )
        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id, # pyright: ignore[reportArgumentType]
                context_id=context.context_id, # pyright: ignore[reportArgumentType]
                status=TaskStatus(
                    state=TaskState.completed #status=TaskStatus(state=TaskState.TASK_STATE_COMPLETED),
                ), 
                final=True
            )
        )

### Streaming and Multiturn